# Yantra — Embedding router (last no-training gain) + guarded eval

Upload to Colab → **Runtime → Run all**. ~10 min total, no GPU training.

**Thesis:** tool choice has been pinned at the lexical ceiling (`expected_name=0.8133` = 244/300, three runs straight). The 56 remaining routing errors are semantic gaps (symptom text → drug names). A tiny embedding model (`all-MiniLM-L6-v2`, 80MB) blended with the proven IDF+char-3gram scorer should recover 10–20 of them → +0.03–0.06 PAS. Routing accuracy is benchmarked **without the LLM** first (gold labels are known), so Cell 3 tells us the gain before we spend eval time.

- **Cell 3:** builds lexical + embedding scorers, sweeps blend weight β, asserts lexical baseline reproduces 244/300, saves winner + tool embeddings.
- **Cell 4:** full GGUF eval with winning router (same fixed-stops harness), guarded promote (only if it beats 0.6747).

Cost: +80MB embedding model at runtime. The DTSA design already puts routing in the runtime, so this is a legitimate system upgrade, not eval hacking.

In [ ]:
# @title 0 — Mount Drive & check prerequisites (no GPU needed)
import os, sys, json, re, math, shutil, time, subprocess, random
from pathlib import Path
from collections import Counter

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_DIR = Path('/content/drive/MyDrive/yantra_run')
except Exception as e:
    RUN_DIR = Path('./yantra_run')
    print('Drive not mounted:', e, '→ using', RUN_DIR)
ART = RUN_DIR / 'artifacts'
print('ART =', ART)

for f in ['toolace_300.jsonl', 'yantra_results.json']:
    print(('OK   ' if (ART/f).exists() else 'MISS ') + f)
print('official PAS =', json.loads((ART/'yantra_results.json').read_text()).get('pas'))

# Official model = SFT2 GGUF (0.6747). DPO2 dirs deliberately excluded (collapsed model).
GGUF_DIRS = [Path('/content/stage8_sft2_q4_gguf'), ART/'stage8_sft2_q4_gguf',
             Path('/content/stage8_sft2_q4'),
             Path('/content/stage6_yantra_q4_gguf'), ART/'stage6_yantra_q4_gguf']
GGUF = None
for d in GGUF_DIRS:
    if d.exists():
        g = sorted(d.glob('*Q4_K_M*.gguf')) or sorted(d.glob('*.gguf'))
        if g:
            GGUF = g[0]; break
print('GGUF =', GGUF if GGUF else 'NONE FOUND → eval cell will error clearly')


In [ ]:
# @title 1 — Installs: sentence-transformers + llama-cpp-python. ~3 min.
import subprocess, sys
def _sh(cmd):
    return subprocess.run(cmd, shell=True).returncode == 0

try:
    import sentence_transformers
    print('sentence_transformers', sentence_transformers.__version__)
except Exception:
    print('installing sentence-transformers ...')
    _sh(f'{sys.executable} -m pip install -q sentence-transformers')
    import sentence_transformers
    print('sentence_transformers', sentence_transformers.__version__)

try:
    import llama_cpp
    print('llama_cpp', llama_cpp.__version__)
except Exception:
    print('installing llama-cpp-python ...')
    if _sh(f'{sys.executable} -m pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121'):
        import llama_cpp
    else:
        _sh(f'{sys.executable} -m pip install -q llama-cpp-python')
        import llama_cpp
    print('llama_cpp', llama_cpp.__version__)

import torch
HAS_CUDA = torch.cuda.is_available()
print('HAS_CUDA =', HAS_CUDA)


In [ ]:
# @title 2 — Shared utils (lexical Router V2 + parsers + metrics, unchanged)
import json, re, math
import numpy as np
from collections import Counter

ACTION_END = '<action_end/>'
DTSA_BIND = re.compile(r'<bind\s+tool="([^"]+)"\s*/>', re.S)
DTSA_ARGS = re.compile(r'<args>(.*?)</args>', re.S)
DTSA_PARAM = re.compile(r'<param\s+name="([^"]+)"\s*>(.*?)</param>', re.S)
def _unesc(v): return v.replace('&amp;','&').replace('&lt;','<').replace('&gt;','>')

def parse_dtsa(text):
    binds = list(DTSA_BIND.finditer(text))
    if not binds: return None
    ends = list(re.finditer(re.escape(ACTION_END), text))
    stopped = bool(ends) and text[ends[-1].end():].strip() == ''
    for j in range(len(binds)-1, -1, -1):
        lb = binds[j]
        seg_end = binds[j+1].start() if j+1 < len(binds) else len(text)
        am = DTSA_ARGS.search(text[lb.end():seg_end])
        if not am: continue
        body = am.group(1); args = {}
        for pm in DTSA_PARAM.finditer(body):
            args[pm.group(1)] = _unesc(pm.group(2).strip())
        if not args:
            for lm in re.finditer(r'name="([^"]+)"\s*>[ \t]*(.*)', body):
                args[lm.group(1)] = _unesc(lm.group(2).strip())
        return {'tool': lb.group(1), 'args': args, 'stopped_clean': stopped}
    return {'tool': binds[0].group(1), 'args': {}, 'stopped_clean': stopped}

def _norm(a):
    if isinstance(a, str):
        s = a.strip()
        if s.lower() in ('true','false'): return s.lower()=='true'
        try: return int(s) if '.' not in s else float(s)
        except ValueError: return s
    return a
def avail_names(tools): return {t.get('function', t).get('name') for t in tools}
def evaluate_case(text, case, mode='dtsa'):
    tools = case['tools']; gold = case['gold']; gold0 = gold[0]
    parsed = parse_dtsa(text)
    if not parsed: return {'parseable':0,'valid_name':0,'expected_name':0,'exact_args':0,'arg_key_overlap':0,'stopped_cleanly':0}
    valid = parsed['tool'] in avail_names(tools)
    expected = parsed['tool'] == gold0['name']
    gk = set(gold0['arguments'].keys()); pk = set(parsed['args'].keys())
    overlap = len(gk & pk)/len(gk) if gk else 1.0
    exact = (set(parsed['args'].keys()) == gk) and all(_norm(gold0['arguments'][k]) == _norm(parsed['args'][k]) for k in gk) if gk else bool(parsed)
    return {'parseable':1,'valid_name':int(valid),'expected_name':int(expected),'exact_args':int(exact),'arg_key_overlap':round(overlap,4),'stopped_cleanly':int(parsed['stopped_clean'])}
def pas(summary, recovery=0.0, multiturn=0.0):
    comps = [summary.get(k,0.0) for k in ('parseable','valid_name','expected_name','exact_args','arg_key_overlap','stopped_cleanly')] + [recovery, multiturn]
    return round(sum(comps)/len(comps), 4)

STOP = ['\n<bind', '<tool_result>', '<user>', '</calls>', '<tool_error>', '<reflect>']

# ---- Lexical Router V2 (byte-identical to the 0.6747 run) ----
toks = lambda s: set(re.findall(r'[a-z0-9_]+', s.lower()))
def cgrams(s, n=3):
    s = re.sub(r'[^a-z0-9 ]','',s.lower())
    return {s[i:i+n] for i in range(max(len(s)-n+1,1))}
print('utils loaded.')


In [ ]:
# @title 3 — Router benchmark: lexical vs embedding vs blend (no LLM, ~2 min)
import numpy as np
from sentence_transformers import SentenceTransformer

cases = [json.loads(l) for l in open(ART/'toolace_300.jsonl') if l.strip()]
print(f'{len(cases)} cases')

# Unique tools across the eval set.
tool_docs = {}
for c in cases:
    for t in c['tools']:
        f = t.get('function', t)
        if f['name'] not in tool_docs:
            tool_docs[f['name']] = f.get('description') or ''
names = sorted(tool_docs)
print(f'{len(names)} unique tools')

# Lexical IDF over name+desc (same construction as eval router).
_docs = [toks(n + ' ' + tool_docs[n]) for n in names]
_df = Counter(w for d in _docs for w in d); _N = max(len(_docs), 1)
_idf = lambda w: math.log(_N / (1 + _df[w]))
_QG_UNUSED = None
def lex_scores(query, tools):
    qt = toks(query); qg = cgrams(query)
    out = {}
    for t in tools:
        nm = t
        d = toks(nm + ' ' + tool_docs[nm])
        s1 = sum(_idf(w) for w in qt & d) / (math.sqrt(sum(_idf(w) for w in d) + 1e-6))
        ng = cgrams(nm)
        s2 = len(qg & ng) / (math.sqrt(len(qg) * len(ng)) + 1)
        out[nm] = s1 + 2.0 * s2
    return out

# Embedding scorer (MiniLM, name+desc per tool).
print('Loading all-MiniLM-L6-v2 ...')
emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
T = emb.encode([n + ' — ' + tool_docs[n] for n in names],
               batch_size=64, show_progress_bar=False, normalize_embeddings=True)
T = np.asarray(T, dtype=np.float32)
Q = emb.encode([c['query'] for c in cases],
               batch_size=64, show_progress_bar=False, normalize_embeddings=True)
Q = np.asarray(Q, dtype=np.float32)
print('embeddings:', T.shape, Q.shape)
np.savez_compressed(ART/'embrouter_tools.npz', names=np.array(names), T=T)
print('saved', ART/'embrouter_tools.npz')

def minmax(v):
    v = np.asarray(v, dtype=float)
    lo, hi = v.min(), v.max()
    return np.zeros_like(v) if hi - lo < 1e-12 else (v - lo) / (hi - lo)

golds = [c['gold'][0]['name'] for c in cases]
COS = Q @ T.T  # [300, n_tools]; column order == names

def acc(beta):
    """Blend: lexical(minmax) + beta * cosine(minmax), per-case argmax."""
    hits, preds = 0, []
    for i, c in enumerate(cases):
        tnames = [t.get('function', t)['name'] for t in c['tools']]
        col = [names.index(n) for n in tnames]
        lex = minmax([lex_scores(c['query'], tnames)[n] for n in tnames])
        em = minmax(COS[i, col])
        best = tnames[int(np.argmax(lex + beta * em))]
        preds.append(best)
        hits += (best == golds[i])
    return hits, preds

base_hits, base_preds = acc(0.0)
print(f'lexical baseline: {base_hits}/300 = {base_hits/300:.4f} (expect 244)')
assert base_hits == 244, f'BASELINE DRIFT: got {base_hits}, expected 244 — stop, router code diverged!'
results = {}
for beta in [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]:
    h, _ = acc(beta)
    results[beta] = h
    print(f'beta={beta:<4} blend: {h}/300 = {h/300:.4f}  ({h-base_hits:+d} vs lexical)')
pure = 0
for i, c in enumerate(cases):
    _best = max(c['tools'], key=lambda t: COS[i, names.index(t.get('function', t)['name'])])
    if _best.get('function', _best)['name'] == golds[i]:
        pure += 1
print(f'pure-embedding (no lexical): {pure}/300 = {pure/300:.4f}')

BEST_BETA = max(results, key=lambda b: (results[b], -b))
print(f'\nWINNER: beta={BEST_BETA} → {results[BEST_BETA]}/300 '
      f'({results[BEST_BETA]-base_hits:+d} vs lexical 244)')
_, win_preds = acc(BEST_BETA)
fixed = [i for i in range(300) if win_preds[i] == golds[i] and base_preds[i] != golds[i]]
broke = [i for i in range(300) if win_preds[i] != golds[i] and base_preds[i] == golds[i]]
print(f'fixed {len(fixed)}, broke {len(broke)}')
for i in fixed[:5]:
    print(f'  FIXED [{i}] gold={golds[i]} query={cases[i]["query"][:90]}')
for i in broke[:5]:
    print(f'  BROKE [{i}] gold={golds[i]} pred={win_preds[i]} query={cases[i]["query"][:90]}')
(ART/'embrouter_winner.json').write_text(json.dumps(
    {'beta': BEST_BETA, 'acc': results[BEST_BETA], 'baseline': base_hits,
     'fixed': fixed, 'broke': broke, 'model': 'sentence-transformers/all-MiniLM-L6-v2'}))
print('saved', ART/'embrouter_winner.json')


In [ ]:
# @title 4 — Full GGUF eval with winning router + GUARDED promote (~5 min)
from llama_cpp import Llama
import time
import numpy as np

# Winner + tool embeddings (reload from disk → restart-safe).
w = json.loads((ART/'embrouter_winner.json').read_text())
BETA = float(w['beta'])
print(f"blend beta={BETA} (routing acc {w['acc']}/300 vs lexical {w['baseline']}/300)")
z = np.load(ART/'embrouter_tools.npz', allow_pickle=True)
NAMES = [str(n) for n in z['names']]
T = np.asarray(z['T'], dtype=np.float32)
DESC = {n: '' for n in NAMES}
cases = [json.loads(l) for l in open(ART/'toolace_300.jsonl') if l.strip()]
for c in cases:
    for t in c['tools']:
        f = t.get('function', t)
        DESC[f['name']] = f.get('description') or ''
_docs = [toks(n + ' ' + DESC[n]) for n in NAMES]
_df = Counter(wd for d in _docs for wd in d); _N = max(len(_docs), 1)
_idf = lambda wd: math.log(_N / (1 + _df[wd]))

from sentence_transformers import SentenceTransformer
emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
Q = np.asarray(emb.encode([c['query'] for c in cases], batch_size=64,
                          show_progress_bar=False, normalize_embeddings=True),
               dtype=np.float32)
COS = Q @ T.T
NAME2COL = {n: j for j, n in enumerate(NAMES)}

def route_emb(i, query, tools):
    tnames = [t.get('function', t)['name'] for t in tools]
    qt = toks(query); qg = cgrams(query)
    lex = []
    for nm in tnames:
        d = toks(nm + ' ' + DESC[nm])
        s1 = sum(_idf(x) for x in qt & d) / (math.sqrt(sum(_idf(x) for x in d) + 1e-6))
        ng = cgrams(nm)
        s2 = len(qg & ng) / (math.sqrt(len(qg) * len(ng)) + 1)
        lex.append(s1 + 2.0 * s2)
    lex = np.asarray(lex, float)
    em = np.asarray([COS[i, NAME2COL[n]] for n in tnames], float)
    lo, hi = lex.min(), lex.max()
    lex_n = np.zeros_like(lex) if hi - lo < 1e-12 else (lex - lo) / (hi - lo)
    lo, hi = em.min(), em.max()
    em_n = np.zeros_like(em) if hi - lo < 1e-12 else (em - lo) / (hi - lo)
    return tnames[int(np.argmax(lex_n + BETA * em_n))]

# Sanity: winner must reproduce Cell-3 accuracy.
sanity = sum(1 for i, c in enumerate(cases)
             if route_emb(i, c['query'], c['tools']) == c['gold'][0]['name'])
print(f'router sanity: {sanity}/300 (expect {w["acc"]})')
assert sanity == w['acc'], f'ROUTER MISMATCH: {sanity} vs {w["acc"]} — stop!'

# GGUF (official SFT2 model first; collapsed DPO2 dirs excluded).
_gg = None
for d in [Path('/content/stage8_sft2_q4_gguf'), ART/'stage8_sft2_q4_gguf',
          Path('/content/stage8_sft2_q4'),
          Path('/content/stage6_yantra_q4_gguf'), ART/'stage6_yantra_q4_gguf']:
    if Path(d).exists():
        g = sorted(Path(d).glob('*Q4_K_M*.gguf')) or sorted(Path(d).glob('*.gguf'))
        if g:
            _gg = g[0]; break
if _gg is None:
    raise FileNotFoundError('No GGUF found — run the SFT2 notebook first.')
if str(_gg).startswith('/content/drive'):
    loc = Path('/tmp/emb_eval.Q4_K_M.gguf')
    if not loc.exists() or loc.stat().st_size != _gg.stat().st_size:
        print('copying GGUF to /tmp ...')
        shutil.copy2(str(_gg), str(loc))
    _gg = loc
print('Evaluating:', _gg)
llm4 = Llama(model_path=str(_gg), n_ctx=4096,
             n_gpu_layers=(-1 if HAS_CUDA else 0), verbose=False)

per = []
t0 = time.time()
for i, case in enumerate(cases):
    tool = route_emb(i, case['query'], case['tools'])
    bind = f'<bind tool="{tool}"/>\n'
    prompt = f"<user>{case['query']}</user>\n<tools>{json.dumps([t.get('function',t) for t in case['tools']], ensure_ascii=False)}</tools>\n<calls>{bind}"
    out = llm4(prompt, max_tokens=768, temperature=0.0, stop=STOP)
    per.append({'id': i, **evaluate_case(bind + (out['choices'][0]['text'] or ''), case)})
    if (i+1) % 50 == 0 or i == len(cases)-1:
        agg_t = {}
        for mm in per:
            for k, v in mm.items():
                if k == 'id': continue
                agg_t[k] = agg_t.get(k, 0.0) + v
        st = {k: round(v/len(per), 4) for k, v in agg_t.items()}
        print(f"  eval {i+1}/{len(cases)}  pas={pas(st):.4f}  expected={st.get('expected_name',0):.3f}  {time.time()-t0:.0f}s", flush=True)

agg = {}
for m in per:
    for k, v in m.items():
        if k == 'id': continue
        agg[k] = agg.get(k, 0.0) + v
summary = {k: round(v/len(cases), 4) for k, v in agg.items()}
score = pas(summary)
(ART/'yantra_results_embrouter.json').write_text(
    json.dumps({'summary': summary, 'pas': score, 'beta': BETA,
                'routing_acc': w['acc'], 'per_case': per}, indent=2))
old = json.loads((ART/'yantra_results.json').read_text())
print('\n' + '='*50)
print(f"CURRENT OFFICIAL: PAS = {old.get('pas')}  {old.get('summary')}")
print(f"EMB-ROUTER CAND.: PAS = {score}  {summary}")
if score > float(old.get('pas', 0)):
    bak = ART/f"yantra_results_backup_{time.strftime('%Y%m%dT%H%M%S')}.json"
    shutil.copy2(str(ART/'yantra_results.json'), str(bak))
    shutil.copy2(str(ART/'yantra_results_embrouter.json'), str(ART/'yantra_results.json'))
    print(f'IMPROVED → promoted (backup: {bak.name})')
else:
    print('NOT better → official KEPT. Nothing overwritten.')
